# Notebook 2 — Environmental feature construction, target seed refinement, and publishable visual analytics

This notebook continues from **Notebook 1: open-source genetic and pathway data collection**.

Notebook 1 produced an open genetic/pathway evidence layer. Notebook 2 builds the **environment + target + integrated modelling table** needed before ML training.

## Main output

The main output is:

`derived/taxon_occurrence_compound_ml_table.tsv`

with one row per:

> **taxon × occurrence × compound**

and feature blocks for:

1. open genetic and pathway evidence;
2. GBIF occurrence and spatial context;
3. optional climate and soil predictors;
4. compound and target-evidence metadata;
5. spatial, taxonomic, compound and source-aware validation groups.

## Scientific novelty made visible

The publishable story is not “we downloaded data.” The novelty is:

> A reproducible open-data framework that links public genetic/pathway potential, global environmental gradients, occurrence space, and medicinal-metabolite evidence to prioritize wild medicinal plants for future pharmacochemical screening and conservation.

Notebook 2 produces manuscript-style figures showing:

1. multi-layer data coverage by taxon;
2. global occurrence-space distribution;
3. taxon–compound evidence heatmap;
4. pathway-potential heatmap;
5. environment–genetic–pathway relationship plots;
6. correlation analysis of engineered features;
7. PCA analysis and PCA biplots;
8. integrated taxon–compound–pathway network;
9. readiness dashboard for selecting taxa and targets for ML.

## Important limitation

This notebook supports **prediction, association, prioritization and hypothesis generation**. It does not prove causal effects of environment on genes or metabolite accumulation.

Causal claims require controlled experiments, common-garden designs, reciprocal transplants, dense population genomics, or wet-lab validation.

## 0. Open-data sources and reproducibility

This notebook expects the output tables of Notebook 1.

Environmental features are implemented in two modes.

**Quick mode**  
Uses GBIF coordinates and reproducible geographic/spatial proxies. This mode runs immediately from Notebook 1 outputs.

**Full environmental mode**  
Optionally extracts WorldClim bioclimatic rasters and SoilGrids variables. Keep this disabled until the quick-mode pipeline works.

Official documentation for relevant open resources:

- WorldClim bioclimatic variables: https://www.worldclim.org/data/bioclim.html
- SoilGrids REST API: https://rest.isric.org/soilgrids/v2.0/docs
- GBIF occurrence API: https://techdocs.gbif.org/en/openapi/v1/occurrence
- scikit-learn PCA: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
- scikit-learn StandardScaler: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

## 1. Recommended environment

```bash
conda create -n medicinal-env-target python=3.11 -y
conda activate medicinal-env-target

conda install -c conda-forge pandas numpy matplotlib scikit-learn scipy networkx requests tqdm pyarrow pyyaml -y
conda install -c conda-forge rasterio geopandas shapely pyproj -y
```

Optional for interactive visualizations:

```bash
pip install plotly kaleido
```

If running in Google Colab, most core packages are already available.

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

import os
import re
import json
import math
import time
import zipfile
import hashlib
import pathlib
import warnings
import datetime as dt
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import pairwise_distances

try:
    from scipy.cluster.hierarchy import linkage, leaves_list
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    import networkx as nx
    NETWORKX_AVAILABLE = True
except Exception:
    NETWORKX_AVAILABLE = False

try:
    import requests
except Exception:
    requests = None

try:
    import rasterio
    RASTERIO_AVAILABLE = True
except Exception:
    RASTERIO_AVAILABLE = False

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

## 2. Configuration

Set `INPUT_DIR` to the folder containing Notebook 1 output files. In Colab, if you upload the `.tsv` files directly, use `/content` or the current working directory.

The default resolver searches common locations automatically.

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

CONFIG = {
    "project_dir": "notebook2_environment_target_outputs",

    # Input search paths. Add your Notebook 1 output folder here if needed.
    "input_dirs": [
        ".",
        "/content",
        "/mnt/data",
        "/content/open_source_genetic_medicinal_value_dataset",
        "/content/open_source_genetic_medicinal_value_dataset/derived",
        "/content/open_source_genetic_medicinal_value_dataset/gbif",
        "/content/open_source_genetic_medicinal_value_dataset/compounds",
        "/content/open_source_genetic_medicinal_value_dataset/ncbi",
    ],

    # Which GBIF table to use for main modelling table.
    # Recommended: "liberal" for broad environmental range; "strict_specimen" for validation.
    "occurrence_mode": "liberal",

    # Maximum occurrences per taxon to keep for first integrated table.
    # Use None for all. Use lower values for fast prototyping.
    "max_occurrences_per_taxon": None,

    # Coordinate and spatial settings.
    "spatial_block_degrees": 5.0,
    "environment_quick_mode": True,

    # Full environmental extraction is optional and disabled by default.
    "worldclim_enabled": False,
    "worldclim_url": "https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_10m_bio.zip",
    "worldclim_variables": [1, 4, 5, 6, 12, 15],
    "soilgrids_enabled": False,
    "soilgrids_max_points": 200,

    # Target logic.
    "curated_positive_pairs": [
        ("TAX_RHODIOLA", "CMP_SALIDROSIDE"),
        ("TAX_RHODIOLA", "CMP_TYROSOL"),
        ("TAX_HYPERICUM", "CMP_HYPERICIN"),
        ("TAX_HYPERICUM", "CMP_TOTAL_FLAVONOIDS"),
        ("TAX_HYPERICUM", "CMP_TOTAL_PHENOLICS"),
        ("TAX_ARTEMISIA_ANNUA", "CMP_ARTEMISININ"),
        ("TAX_GLYCYRRHIZA_GLABRA", "CMP_GLYCYRRHIZIN"),
        ("TAX_SEDUM", "CMP_TOTAL_FLAVONOIDS"),
        ("TAX_SEDUM", "CMP_TOTAL_PHENOLICS"),
        ("TAX_TAXUS", "CMP_PACLITAXEL"),
    ],

    # Use family rows only as hierarchy/context, not regular ML samples.
    "exclude_hierarchy_context_from_ml_table": True,

    # Figures.
    "figure_dpi": 300,
    "save_pdf": True,

    "random_seed": 42,
}

PROJECT = pathlib.Path(CONFIG["project_dir"]).resolve()
DIRS = {
    "derived": PROJECT / "derived",
    "figures": PROJECT / "figures",
    "tables": PROJECT / "tables",
    "external": PROJECT / "external",
    "logs": PROJECT / "logs",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
CONFIG["run_id"] = RUN_ID
CONFIG["created_utc"] = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")

with open(PROJECT / "notebook2_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

PROJECT

In [ ]:
# ============================================================
# 3. Utility functions
# ============================================================

def safe_name(x: str, max_len: int = 160) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(x)).strip("_")[:max_len]

def sha256_file(path: pathlib.Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def find_input_file(filename: str) -> pathlib.Path:
    candidates = []
    for d in CONFIG["input_dirs"]:
        d = pathlib.Path(d)
        candidates.append(d / filename)
        candidates.extend(d.rglob(filename) if d.exists() and d.is_dir() else [])
    existing = [p for p in candidates if p.exists()]
    if not existing:
        raise FileNotFoundError(f"Could not find {filename}. Add its folder to CONFIG['input_dirs'].")
    # Prefer shortest path / direct upload.
    existing = sorted(set(existing), key=lambda p: (len(str(p)), str(p)))
    return existing[0]

def load_tsv(filename: str, required: bool = True) -> pd.DataFrame:
    try:
        path = find_input_file(filename)
        df = pd.read_csv(path, sep="\t", dtype=str, low_memory=False)
        print(f"Loaded {filename}: {df.shape} from {path}")
        return df
    except Exception as e:
        if required:
            raise
        print(f"Optional file not loaded: {filename} ({e})")
        return pd.DataFrame()

def write_table(df: pd.DataFrame, path_stem: pathlib.Path) -> Dict[str, str]:
    path_stem.parent.mkdir(parents=True, exist_ok=True)
    tsv = path_stem.with_suffix(".tsv")
    df.to_csv(tsv, sep="\t", index=False)
    out = {"tsv": str(tsv)}
    try:
        pq = path_stem.with_suffix(".parquet")
        df.to_parquet(pq, index=False)
        out["parquet"] = str(pq)
    except Exception as e:
        print(f"Parquet skipped for {path_stem.name}: {e}")
    return out

def save_figure(fig, name: str):
    png_path = DIRS["figures"] / f"{safe_name(name)}.png"
    fig.savefig(png_path, dpi=CONFIG["figure_dpi"], bbox_inches="tight")
    outputs = [str(png_path)]
    if CONFIG["save_pdf"]:
        pdf_path = DIRS["figures"] / f"{safe_name(name)}.pdf"
        fig.savefig(pdf_path, bbox_inches="tight")
        outputs.append(str(pdf_path))
    print("Saved:", outputs)
    return outputs

def to_numeric_series(s):
    return pd.to_numeric(s, errors="coerce")

def minmax(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

def log1p_numeric(s):
    return np.log1p(pd.to_numeric(s, errors="coerce").fillna(0))

def add_panel_label(ax, label):
    ax.text(-0.08, 1.05, label, transform=ax.transAxes, fontsize=13, fontweight="bold", va="top")

## 4. Load Notebook 1 handoff tables

In [ ]:
# ============================================================
# 4. Load Notebook 1 handoff tables
# ============================================================

open_genetic_df = load_tsv("open_genetic_feature_matrix_by_taxon_v2.tsv")
priority_df = load_tsv("ml_taxon_priority_table.tsv")
pathway_df = load_tsv("pathway_features_by_taxon.tsv")
target_seed_df = load_tsv("taxon_compound_target_seed_table.tsv")
compound_df = load_tsv("compound_dictionary.tsv")
candidate_gene_df = load_tsv("candidate_gene_evidence_tiered.tsv", required=False)
assemblies_df = load_tsv("ncbi_assemblies_manifest.tsv", required=False)

if CONFIG["occurrence_mode"] == "strict_specimen":
    occurrence_df = load_tsv("gbif_occurrence_strict_specimen.tsv")
elif CONFIG["occurrence_mode"] == "strict":
    occurrence_df = load_tsv("gbif_occurrence_strict_filtered.tsv", required=False)
    if occurrence_df.empty:
        occurrence_df = load_tsv("gbif_occurrence_strict_specimen.tsv")
else:
    occurrence_df = load_tsv("gbif_occurrence_liberal_filtered.tsv")

# Basic type conversions
for df in [open_genetic_df, priority_df, pathway_df, target_seed_df, compound_df, occurrence_df]:
    if "taxon_id" in df.columns:
        df["taxon_id"] = df["taxon_id"].astype(str)

print("\nInput row counts:")
display(pd.DataFrame([
    {"table": "open_genetic_feature_matrix", "rows": len(open_genetic_df), "columns": open_genetic_df.shape[1]},
    {"table": "ml_taxon_priority", "rows": len(priority_df), "columns": priority_df.shape[1]},
    {"table": "pathway_features", "rows": len(pathway_df), "columns": pathway_df.shape[1]},
    {"table": "target_seed", "rows": len(target_seed_df), "columns": target_seed_df.shape[1]},
    {"table": "compound_dictionary", "rows": len(compound_df), "columns": compound_df.shape[1]},
    {"table": "candidate_gene_evidence", "rows": len(candidate_gene_df), "columns": candidate_gene_df.shape[1] if not candidate_gene_df.empty else 0},
    {"table": "assemblies", "rows": len(assemblies_df), "columns": assemblies_df.shape[1] if not assemblies_df.empty else 0},
    {"table": f"occurrence_{CONFIG['occurrence_mode']}", "rows": len(occurrence_df), "columns": occurrence_df.shape[1]},
]))

## 5. Input QC and manuscript-ready coverage summary

This creates a compact coverage table used in later figures and in the manuscript methods/results.

In [ ]:
# ============================================================
# 5. Input QC and coverage summary
# ============================================================

coverage_cols = [
    "taxon_id", "scientific_name", "family", "rank", "ml_role",
    "open_data_ml_readiness", "n_public_barcode_records",
    "n_chloroplast_plastid_records", "n_public_genome_assemblies",
    "n_ena_rnaseq_runs", "n_pathways_with_specific_candidates",
    "sum_pathway_potential_score", "n_taxon_compound_pairs_with_literature",
    "n_taxon_compound_pairs_for_quant_curation",
]
coverage_cols = [c for c in coverage_cols if c in open_genetic_df.columns]
coverage_df = open_genetic_df[coverage_cols].copy()

numeric_cov_cols = [c for c in coverage_df.columns if c.startswith("n_") or c.endswith("_score")]
for c in numeric_cov_cols:
    coverage_df[c] = pd.to_numeric(coverage_df[c], errors="coerce").fillna(0)

# Add occurrence counts
occ_counts = occurrence_df.groupby("taxon_id").size().reset_index(name="n_occurrence_records_selected")
coverage_df = coverage_df.merge(occ_counts, on="taxon_id", how="left")
coverage_df["n_occurrence_records_selected"] = coverage_df["n_occurrence_records_selected"].fillna(0).astype(int)

# Candidate vs hierarchy status
coverage_df["ordinary_ml_candidate"] = ~coverage_df.get("open_data_ml_readiness", "").isin(["hierarchy_context"])
if CONFIG["exclude_hierarchy_context_from_ml_table"] and "open_data_ml_readiness" in coverage_df.columns:
    coverage_df.loc[coverage_df["open_data_ml_readiness"].eq("hierarchy_context"), "ordinary_ml_candidate"] = False

write_table(coverage_df, DIRS["derived"] / "notebook2_taxon_coverage_summary")
coverage_df

## 6. Refine taxon–compound target seed table

Notebook 1 produced a target seed table from PubMed search results. Here we convert it into a conservative target-evidence scale.

This is **not yet a final metabolite concentration label**. It is a reproducible target-evidence score for open-data prioritization.

Target evidence scale:

- `0`: no usable evidence in current open seed table;
- `1`: weak/indirect or noisy literature context;
- `2`: literature support with compound/title match;
- `3`: curated high-confidence positive pair from project strategy.

Missing literature is not true absence. Therefore, rows with target evidence `0` should be treated as **background or pseudo-negative candidates**, not confirmed negatives.

In [ ]:
# ============================================================
# 6. Refine target-evidence table
# ============================================================

target_df = target_seed_df.copy()

for c in ["n_supporting_pubmed_records", "n_recent_pubmed_records_2015plus", "has_literature_support", "has_exact_compound_in_title", "has_accumulation_or_pathway_terms"]:
    if c in target_df.columns:
        target_df[c] = pd.to_numeric(target_df[c], errors="coerce").fillna(0)

curated_pairs = set(CONFIG["curated_positive_pairs"])
target_df["is_curated_positive_pair"] = target_df.apply(lambda r: (r["taxon_id"], r["compound_id"]) in curated_pairs, axis=1).astype(int)

def target_evidence_score(row):
    if row.get("is_curated_positive_pair", 0) == 1:
        return 3
    rec = str(row.get("target_type_recommendation", ""))
    if rec == "candidate_for_quantitative_curation":
        return 2
    if rec == "binary_presence_or_literature_support":
        return 2
    if rec == "literature_noise_or_indirect_context":
        return 1
    if row.get("has_literature_support", 0) > 0:
        return 1
    return 0

target_df["target_evidence_score"] = target_df.apply(target_evidence_score, axis=1)
target_df["target_evidence_class"] = target_df["target_evidence_score"].map({
    0: "no_usable_evidence",
    1: "weak_or_indirect",
    2: "literature_supported",
    3: "curated_strong_positive",
})

target_df["target_label_for_open_data_classifier"] = np.where(target_df["target_evidence_score"] >= 2, 1, 0)
target_df["pseudo_negative_flag"] = np.where(target_df["target_evidence_score"] == 0, 1, 0)
target_df["target_warning"] = np.where(
    target_df["pseudo_negative_flag"].eq(1),
    "Missing evidence is not confirmed absence; use as pseudo-negative/background only.",
    ""
)

write_table(target_df, DIRS["derived"] / "taxon_compound_target_evidence_refined")
target_df.sort_values(["target_evidence_score", "n_supporting_pubmed_records"], ascending=[False, False]).head(30)

## 7. Reshape pathway features to wide format

This creates explicit pathway feature columns for the integrated ML table and for correlation/PCA analyses.

In [ ]:
# ============================================================
# 7. Pathway features wide table
# ============================================================

pathway_numeric_cols = [
    "n_candidate_protein_hits",
    "n_specific_candidate_hits",
    "n_broad_family_hits",
    "max_specificity_score",
    "pathway_potential_score",
]
for c in pathway_numeric_cols:
    if c in pathway_df.columns:
        pathway_df[c] = pd.to_numeric(pathway_df[c], errors="coerce").fillna(0)

wide_parts = []
for value_col in pathway_numeric_cols:
    if value_col in pathway_df.columns:
        tmp = pathway_df.pivot_table(index="taxon_id", columns="pathway_id", values=value_col, aggfunc="max", fill_value=0)
        tmp.columns = [f"{safe_name(p)}__{value_col}" for p in tmp.columns]
        wide_parts.append(tmp)

if wide_parts:
    pathway_wide_df = pd.concat(wide_parts, axis=1).reset_index()
else:
    pathway_wide_df = pd.DataFrame(columns=["taxon_id"])

write_table(pathway_wide_df, DIRS["derived"] / "pathway_features_by_taxon_wide")
pathway_wide_df.head()

## 8. Occurrence feature engineering and spatial validation groups

Quick-mode environmental features are not substitutes for real climate variables, but they provide reproducible spatial proxies and allow the end-to-end dataset to be built immediately.

Notebook 2 also includes optional full environmental extraction cells later.

In [ ]:
# ============================================================
# 8. Occurrence feature engineering
# ============================================================

occ_df = occurrence_df.copy()

# Numeric conversion
for c in ["decimal_latitude", "decimal_longitude", "coordinate_uncertainty_m"]:
    if c in occ_df.columns:
        occ_df[c] = pd.to_numeric(occ_df[c], errors="coerce")

occ_df = occ_df.dropna(subset=["decimal_latitude", "decimal_longitude"]).copy()
occ_df = occ_df[
    occ_df["decimal_latitude"].between(-90, 90) &
    occ_df["decimal_longitude"].between(-180, 180)
].copy()

# Optional occurrence downsampling per taxon for quick prototyping.
if CONFIG["max_occurrences_per_taxon"] is not None:
    occ_df = (
        occ_df.groupby("taxon_id", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), CONFIG["max_occurrences_per_taxon"]), random_state=CONFIG["random_seed"]))
        .reset_index(drop=True)
    )

# Geographic/spatial features
block = float(CONFIG["spatial_block_degrees"])
occ_df["abs_latitude"] = occ_df["decimal_latitude"].abs()
occ_df["latitude_squared"] = occ_df["decimal_latitude"] ** 2
occ_df["longitude_squared"] = occ_df["decimal_longitude"] ** 2
occ_df["sin_latitude"] = np.sin(np.deg2rad(occ_df["decimal_latitude"]))
occ_df["cos_latitude"] = np.cos(np.deg2rad(occ_df["decimal_latitude"]))
occ_df["sin_longitude"] = np.sin(np.deg2rad(occ_df["decimal_longitude"]))
occ_df["cos_longitude"] = np.cos(np.deg2rad(occ_df["decimal_longitude"]))
occ_df["hemisphere"] = np.where(occ_df["decimal_latitude"] >= 0, "north", "south")
occ_df["spatial_block_lat"] = np.floor(occ_df["decimal_latitude"] / block).astype("Int64")
occ_df["spatial_block_lon"] = np.floor(occ_df["decimal_longitude"] / block).astype("Int64")
occ_df["spatial_block_id"] = occ_df["spatial_block_lat"].astype(str) + "_" + occ_df["spatial_block_lon"].astype(str)

# Latitudinal ecological bands as coarse environmental strata
bins = [-90, -60, -35, -23.5, 0, 23.5, 35, 60, 90]
labels = ["antarctic_subpolar", "south_temperate", "south_subtropical", "south_tropical",
          "north_tropical", "north_subtropical", "north_temperate", "arctic_subpolar"]
occ_df["latitudinal_band"] = pd.cut(occ_df["decimal_latitude"], bins=bins, labels=labels, include_lowest=True)

# Record-quality flags
occ_df["coordinate_uncertainty_m"] = pd.to_numeric(occ_df.get("coordinate_uncertainty_m", np.nan), errors="coerce")
occ_df["coordinate_uncertainty_missing"] = occ_df["coordinate_uncertainty_m"].isna().astype(int)
occ_df["coordinate_uncertainty_log1p"] = np.log1p(occ_df["coordinate_uncertainty_m"].fillna(0))
occ_df["basis_of_record"] = occ_df.get("basis_of_record", "").fillna("unknown").astype(str)

write_table(occ_df, DIRS["derived"] / "occurrence_features_quick_mode")
occ_df.head()

## 9. Optional full environmental extraction: WorldClim

This cell is disabled by default. Enable it only after the quick-mode pipeline works.

It downloads WorldClim 2.1 10-minute bioclimatic rasters and extracts selected variables at GBIF coordinates.

Recommended variables for first publication-grade feature set:

- BIO1: annual mean temperature
- BIO4: temperature seasonality
- BIO5: max temperature of warmest month
- BIO6: min temperature of coldest month
- BIO12: annual precipitation
- BIO15: precipitation seasonality

In [ ]:
# ============================================================
# 9. Optional WorldClim extraction
# ============================================================

def download_file(url: str, out_path: pathlib.Path, chunk_size: int = 1024 * 1024):
    if requests is None:
        raise ImportError("requests is required for downloads.")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        print("Already exists:", out_path)
        return out_path
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return out_path

def extract_worldclim_variables(occ_df: pd.DataFrame, variables: List[int]) -> pd.DataFrame:
    if not RASTERIO_AVAILABLE:
        raise ImportError("rasterio is required for WorldClim extraction.")
    worldclim_dir = DIRS["external"] / "worldclim"
    zip_path = worldclim_dir / "wc2.1_10m_bio.zip"
    download_file(CONFIG["worldclim_url"], zip_path)
    extract_dir = worldclim_dir / "wc2.1_10m_bio"
    extract_dir.mkdir(parents=True, exist_ok=True)
    if not list(extract_dir.glob("*.tif")):
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)

    out = occ_df[["gbif_key", "decimal_longitude", "decimal_latitude"]].copy()
    coords = list(zip(out["decimal_longitude"], out["decimal_latitude"]))

    for var in variables:
        pattern = f"wc2.1_10m_bio_{var}.tif"
        matches = list(extract_dir.rglob(pattern))
        if not matches:
            print("Raster not found:", pattern)
            continue
        raster_path = matches[0]
        with rasterio.open(raster_path) as src:
            vals = [v[0] if len(v) else np.nan for v in src.sample(coords)]
        out[f"worldclim_bio{var}"] = vals
        print("Extracted", f"BIO{var}", "from", raster_path)

    return out

if CONFIG["worldclim_enabled"]:
    worldclim_df = extract_worldclim_variables(occ_df, CONFIG["worldclim_variables"])
    write_table(worldclim_df, DIRS["derived"] / "worldclim_features_by_occurrence")
else:
    worldclim_df = pd.DataFrame()
    print("WorldClim extraction disabled. Set CONFIG['worldclim_enabled'] = True to run.")

## 10. Optional full environmental extraction: SoilGrids point queries

This cell is disabled by default because point-wise API requests can be slow. Use it for a reduced set of high-priority occurrences or after deduplicating spatial blocks.

In [ ]:
# ============================================================
# 10. Optional SoilGrids extraction
# ============================================================

def query_soilgrids_point(lon: float, lat: float, properties=("phh2o", "soc"), depth="0-5cm") -> Dict:
    if requests is None:
        raise ImportError("requests is required for SoilGrids.")
    url = "https://rest.isric.org/soilgrids/v2.0/properties/query"
    params = {"lon": lon, "lat": lat}
    for p in properties:
        params.setdefault("property", [])
    # requests handles repeated params with list of tuples
    params_list = [("lon", lon), ("lat", lat)]
    for p in properties:
        params_list.append(("property", p))
    params_list.extend([("depth", depth), ("value", "mean")])
    r = requests.get(url, params=params_list, timeout=60)
    r.raise_for_status()
    data = r.json()
    out = {}
    layers = data.get("properties", {}).get("layers", [])
    for layer in layers:
        name = layer.get("name")
        for d in layer.get("depths", []):
            if d.get("label") == depth:
                vals = d.get("values", {})
                out[f"soilgrids_{name}_{depth}_mean"] = vals.get("mean")
    return out

if CONFIG["soilgrids_enabled"]:
    soil_rows = []
    # Query only one representative point per spatial block to reduce calls.
    soil_points = occ_df.drop_duplicates(subset=["spatial_block_id"]).head(CONFIG["soilgrids_max_points"]).copy()
    for _, r in soil_points.iterrows():
        try:
            vals = query_soilgrids_point(float(r["decimal_longitude"]), float(r["decimal_latitude"]))
            vals["spatial_block_id"] = r["spatial_block_id"]
            vals["decimal_longitude"] = r["decimal_longitude"]
            vals["decimal_latitude"] = r["decimal_latitude"]
            soil_rows.append(vals)
            time.sleep(0.2)
        except Exception as e:
            soil_rows.append({
                "spatial_block_id": r["spatial_block_id"],
                "decimal_longitude": r["decimal_longitude"],
                "decimal_latitude": r["decimal_latitude"],
                "soilgrids_error": str(e)
            })
    soilgrids_df = pd.DataFrame(soil_rows)
    write_table(soilgrids_df, DIRS["derived"] / "soilgrids_features_by_spatial_block")
else:
    soilgrids_df = pd.DataFrame()
    print("SoilGrids extraction disabled. Set CONFIG['soilgrids_enabled'] = True to run.")

## 11. Construct the integrated taxon–occurrence–compound ML table

The modelling unit is:

> taxon × occurrence × compound

Genetic/pathway features attach at taxon level.  
Occurrence and environment features attach at occurrence level.  
Target-evidence features attach at taxon–compound level.

This design makes the open-data constraint explicit and avoids pretending that public genetic records are exact population genotypes.

In [ ]:
# ============================================================
# 11. Build integrated ML table
# ============================================================

# Candidate taxa for ordinary ML table
taxon_meta_cols = [
    "taxon_id", "scientific_name", "family", "rank", "priority", "ml_role",
    "open_data_ml_readiness", "genetic_data_availability_score",
    "pathway_support_score", "n_public_barcode_records",
    "n_chloroplast_plastid_records", "n_public_genome_assemblies",
    "n_ena_rnaseq_runs", "n_pathways_with_specific_candidates",
    "sum_pathway_potential_score",
]
taxon_meta_cols = [c for c in taxon_meta_cols if c in open_genetic_df.columns]
taxon_features = open_genetic_df[taxon_meta_cols].copy()

if CONFIG["exclude_hierarchy_context_from_ml_table"] and "open_data_ml_readiness" in taxon_features.columns:
    taxon_features = taxon_features[~taxon_features["open_data_ml_readiness"].eq("hierarchy_context")].copy()

# Add pathway wide features
taxon_features = taxon_features.merge(pathway_wide_df, on="taxon_id", how="left")

# Occurrence rows only for candidate taxa
occ_ml = occ_df[occ_df["taxon_id"].isin(taxon_features["taxon_id"])].copy()

# Join taxon-level genetic/pathway features to occurrences
occ_taxon = occ_ml.merge(taxon_features, on="taxon_id", how="left", suffixes=("", "_taxon"))

# Cross with target seed table for the same taxon
target_cols = [
    "taxon_id", "compound_id", "compound_name", "compound_class", "compound_record_type",
    "n_supporting_pubmed_records", "n_recent_pubmed_records_2015plus",
    "has_literature_support", "has_accumulation_or_pathway_terms",
    "has_exact_compound_in_title", "target_type_recommendation",
    "target_confidence_initial", "is_curated_positive_pair",
    "target_evidence_score", "target_evidence_class",
    "target_label_for_open_data_classifier", "pseudo_negative_flag", "target_warning",
]
target_cols = [c for c in target_cols if c in target_df.columns]
target_for_join = target_df[target_cols].copy()

ml_table = occ_taxon.merge(target_for_join, on="taxon_id", how="inner")

# Join compound dictionary metadata
compound_cols = [
    "compound_id", "record_type", "pubchem_cid", "molecular_formula",
    "molecular_weight", "inchikey", "compound_dictionary_status"
]
compound_cols = [c for c in compound_cols if c in compound_df.columns]
ml_table = ml_table.merge(compound_df[compound_cols], on="compound_id", how="left", suffixes=("", "_compound_dict"))

# Add optional WorldClim
if not worldclim_df.empty and "gbif_key" in ml_table.columns:
    ml_table = ml_table.merge(worldclim_df, on="gbif_key", how="left")

# Add optional SoilGrids by spatial block
if not soilgrids_df.empty and "spatial_block_id" in ml_table.columns:
    ml_table = ml_table.merge(soilgrids_df, on="spatial_block_id", how="left")

# Validation groups
ml_table["taxonomic_group_id"] = ml_table.get("family", "").astype(str) + "__" + ml_table.get("taxon_id", "").astype(str)
ml_table["compound_group_id"] = ml_table.get("compound_class", "").astype(str)
ml_table["spatial_group_id"] = ml_table.get("spatial_block_id", "").astype(str)
ml_table["source_group_id"] = CONFIG["occurrence_mode"] + "__notebook1_open_sources"

# Stable row ID
ml_table = ml_table.reset_index(drop=True)
ml_table["ml_row_id"] = ["MLROW_" + str(i).zfill(9) for i in range(len(ml_table))]

# Place row id first
cols = ["ml_row_id"] + [c for c in ml_table.columns if c != "ml_row_id"]
ml_table = ml_table[cols]

write_table(ml_table, DIRS["derived"] / "taxon_occurrence_compound_ml_table")

print("Integrated ML table:", ml_table.shape)
ml_table.head()

## 12. Publication Figure 1 — Multi-layer open-data coverage by taxon

This figure shows the central resource contribution: which taxa have usable public evidence across occurrence, genome, barcode, transcriptome, pathway and metabolite-literature layers.

In [ ]:
# ============================================================
# 12. Figure 1: Data coverage dashboard
# ============================================================

fig_df = coverage_df.copy()

plot_cols = [
    "n_occurrence_records_selected",
    "n_public_barcode_records",
    "n_chloroplast_plastid_records",
    "n_public_genome_assemblies",
    "n_ena_rnaseq_runs",
    "n_pathways_with_specific_candidates",
    "n_taxon_compound_pairs_with_literature",
]
plot_cols = [c for c in plot_cols if c in fig_df.columns]

for c in plot_cols:
    fig_df[c] = pd.to_numeric(fig_df[c], errors="coerce").fillna(0)

fig_df["sort_score"] = fig_df[plot_cols].apply(lambda x: np.log1p(x).sum(), axis=1)
fig_df = fig_df.sort_values("sort_score", ascending=True)

matrix = fig_df[plot_cols].apply(lambda x: np.log1p(x))
matrix_norm = matrix.apply(minmax, axis=0)

fig, ax = plt.subplots(figsize=(12, max(5, 0.45 * len(fig_df))))
im = ax.imshow(matrix_norm.values, aspect="auto", interpolation="nearest")

ax.set_yticks(range(len(fig_df)))
ax.set_yticklabels(fig_df["scientific_name"].astype(str))
ax.set_xticks(range(len(plot_cols)))
ax.set_xticklabels([c.replace("n_", "").replace("_", " ") for c in plot_cols], rotation=45, ha="right")

for i in range(len(fig_df)):
    for j, c in enumerate(plot_cols):
        raw = int(fig_df.iloc[i][c])
        ax.text(j, i, str(raw), ha="center", va="center", fontsize=8)

ax.set_title("Open-data evidence coverage by taxon")
ax.set_xlabel("Evidence layer")
ax.set_ylabel("Taxon")
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Column-normalized log evidence count")

save_figure(fig, "figure1_multilayer_open_data_coverage_by_taxon")
plt.show()

## 13. Publication Figure 2 — Global occurrence-space map

This figure demonstrates the environmental-gradient component of the study. It is intentionally not a species distribution model; it is a transparent visualization of occurrence space used for environmental extraction and validation grouping.

In [ ]:
# ============================================================
# 13. Figure 2: Occurrence-space map
# ============================================================

map_df = occ_df.copy()
taxa_to_plot = map_df["taxon_id"].value_counts().index.tolist()
taxon_to_name = open_genetic_df.set_index("taxon_id")["scientific_name"].to_dict() if "scientific_name" in open_genetic_df else {}

fig, ax = plt.subplots(figsize=(13, 6.5))

# Simple global extent. Avoid basemap dependency for reproducibility.
for taxon_id in taxa_to_plot:
    sub = map_df[map_df["taxon_id"] == taxon_id]
    ax.scatter(
        sub["decimal_longitude"],
        sub["decimal_latitude"],
        s=8,
        alpha=0.35,
        label=taxon_to_name.get(taxon_id, taxon_id),
        linewidths=0
    )

ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xticks(np.arange(-180, 181, 60))
ax.set_yticks(np.arange(-90, 91, 30))
ax.grid(True, linewidth=0.4, alpha=0.4)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Global occurrence space used for environmental feature construction ({CONFIG['occurrence_mode']} GBIF set)")

# Keep legend compact
if len(taxa_to_plot) <= 12:
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=8)

save_figure(fig, "figure2_global_occurrence_space")
plt.show()

## 14. Publication Figure 3 — Taxon–compound evidence heatmap

This figure makes the medicinal-metabolite target layer explicit and helps avoid overclaiming. The heatmap shows evidence strength, not verified concentration.

In [ ]:
# ============================================================
# 14. Figure 3: Taxon-compound evidence heatmap
# ============================================================

heat_df = target_df.copy()
# Use non-hierarchy rows by default
if "ml_role" in heat_df.columns:
    heat_df_plot = heat_df[~heat_df["ml_role"].eq("hierarchy_context")].copy()
else:
    heat_df_plot = heat_df.copy()

pivot = heat_df_plot.pivot_table(
    index="scientific_name",
    columns="compound_name",
    values="target_evidence_score",
    aggfunc="max",
    fill_value=0
)

# Sort rows by total evidence
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=True).index]

fig, ax = plt.subplots(figsize=(12, max(4.5, 0.5 * len(pivot))))
im = ax.imshow(pivot.values, aspect="auto", interpolation="nearest", vmin=0, vmax=3)

ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha="right")

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = int(pivot.iloc[i, j])
        label = "" if val == 0 else str(val)
        ax.text(j, i, label, ha="center", va="center", fontsize=9)

ax.set_title("Taxon–compound open evidence score")
ax.set_xlabel("Compound or compound class")
ax.set_ylabel("Taxon")
cbar = fig.colorbar(im, ax=ax, shrink=0.8, ticks=[0,1,2,3])
cbar.ax.set_yticklabels(["0 none", "1 weak", "2 literature", "3 curated"])

save_figure(fig, "figure3_taxon_compound_evidence_heatmap")
plt.show()

## 15. Publication Figure 4 — Pathway-potential heatmap

This figure links public genetic/pathway evidence to medicinal compound classes.

In [ ]:
# ============================================================
# 15. Figure 4: Pathway potential heatmap
# ============================================================

pwy_plot = pathway_df.copy()
for c in ["pathway_potential_score", "n_specific_candidate_hits", "n_broad_family_hits"]:
    if c in pwy_plot:
        pwy_plot[c] = pd.to_numeric(pwy_plot[c], errors="coerce").fillna(0)

# Join taxon names and ML role
tax_meta = open_genetic_df[["taxon_id", "scientific_name", "open_data_ml_readiness"]].drop_duplicates()
pwy_plot = pwy_plot.merge(tax_meta, on="taxon_id", how="left")

pwy_pivot = pwy_plot.pivot_table(
    index="scientific_name",
    columns="pathway_id",
    values="pathway_potential_score",
    aggfunc="max",
    fill_value=0
)

pwy_pivot = pwy_pivot.loc[pwy_pivot.sum(axis=1).sort_values(ascending=True).index]
pwy_values = np.log1p(pwy_pivot)

fig, ax = plt.subplots(figsize=(11, max(4.5, 0.5 * len(pwy_pivot))))
im = ax.imshow(pwy_values.values, aspect="auto", interpolation="nearest")

ax.set_yticks(range(len(pwy_pivot.index)))
ax.set_yticklabels(pwy_pivot.index)
ax.set_xticks(range(len(pwy_pivot.columns)))
ax.set_xticklabels([c.replace("PWY_", "") for c in pwy_pivot.columns], rotation=45, ha="right")

for i in range(pwy_pivot.shape[0]):
    for j in range(pwy_pivot.shape[1]):
        raw = pwy_pivot.iloc[i, j]
        if raw > 0:
            ax.text(j, i, f"{raw:.1f}", ha="center", va="center", fontsize=8)

ax.set_title("Open genetic/pathway potential by taxon")
ax.set_xlabel("Pathway")
ax.set_ylabel("Taxon")
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("log1p(pathway potential score)")

save_figure(fig, "figure4_pathway_potential_heatmap")
plt.show()

## 16. Publication Figure 5 — Evidence-prioritization quadrant

This figure is useful for a manuscript because it shows how taxa are prioritized based on independent evidence axes:

- genetic data availability;
- pathway support;
- metabolite/literature support.

Taxa in the upper-right quadrant are best suited for the first ML prototype and later manual curation.

In [ ]:
# ============================================================
# 16. Figure 5: Evidence-prioritization quadrant
# ============================================================

quad_df = open_genetic_df.copy()
for c in ["genetic_data_availability_score", "pathway_support_score", "n_taxon_compound_pairs_with_literature"]:
    if c in quad_df.columns:
        quad_df[c] = pd.to_numeric(quad_df[c], errors="coerce").fillna(0)

fig, ax = plt.subplots(figsize=(9, 7))

readiness = quad_df.get("open_data_ml_readiness", pd.Series(["unknown"] * len(quad_df)))
unique_classes = readiness.fillna("unknown").unique().tolist()

for cls in unique_classes:
    sub = quad_df[readiness.fillna("unknown") == cls]
    sizes = 40 + 25 * np.log1p(sub.get("n_taxon_compound_pairs_with_literature", pd.Series([0]*len(sub))).astype(float))
    ax.scatter(
        sub["genetic_data_availability_score"],
        sub["pathway_support_score"],
        s=sizes,
        alpha=0.75,
        label=str(cls)
    )

for _, r in quad_df.iterrows():
    ax.text(
        r["genetic_data_availability_score"] + 0.05,
        r["pathway_support_score"] + 0.05,
        str(r["scientific_name"]),
        fontsize=8
    )

ax.axvline(quad_df["genetic_data_availability_score"].median(), linestyle="--", linewidth=0.8)
ax.axhline(quad_df["pathway_support_score"].median(), linestyle="--", linewidth=0.8)
ax.set_xlabel("Genetic data availability score")
ax.set_ylabel("Pathway support score")
ax.set_title("Prioritization of taxa by open genetic and pathway evidence")
ax.legend(frameon=False, loc="best")

save_figure(fig, "figure5_evidence_prioritization_quadrant")
plt.show()

## 17. Feature relationship analysis

This section creates a numeric feature matrix and analyzes relationships among open genetic, pathway, occurrence, and target-evidence features.

It includes:

1. Pearson correlation;
2. Spearman correlation;
3. clustered correlation heatmap;
4. feature–target correlation ranking;
5. pairwise relationship plots for the strongest features.

These results are exploratory and should be reported as feature associations, not causal mechanisms.

In [ ]:
# ============================================================
# 17A. Build feature matrix for correlation/PCA
# ============================================================

# Aggregate integrated table to taxon-compound level to reduce pseudoreplication from repeated occurrences.
agg_numeric = {
    "target_evidence_score": "max",
    "target_label_for_open_data_classifier": "max",
    "n_supporting_pubmed_records": "max",
    "genetic_data_availability_score": "max",
    "pathway_support_score": "max",
    "n_public_barcode_records": "max",
    "n_chloroplast_plastid_records": "max",
    "n_public_genome_assemblies": "max",
    "n_ena_rnaseq_runs": "max",
    "n_pathways_with_specific_candidates": "max",
    "sum_pathway_potential_score": "max",
    "decimal_latitude": "mean",
    "decimal_longitude": "mean",
    "abs_latitude": "mean",
    "latitude_squared": "mean",
    "coordinate_uncertainty_log1p": "mean",
}
agg_numeric = {k:v for k,v in agg_numeric.items() if k in ml_table.columns}

group_cols = ["taxon_id", "scientific_name", "compound_id", "compound_name", "compound_class", "target_evidence_class"]
group_cols = [c for c in group_cols if c in ml_table.columns]

taxon_compound_feature_df = ml_table.groupby(group_cols, dropna=False).agg(agg_numeric).reset_index()

# Add pathway-wide columns aggregated at taxon level
path_cols = [c for c in pathway_wide_df.columns if c != "taxon_id"]
taxon_compound_feature_df = taxon_compound_feature_df.merge(pathway_wide_df, on="taxon_id", how="left")

# Add occurrence diversity by taxon
occ_summary = occ_df.groupby("taxon_id").agg(
    n_occurrences=("gbif_key", "count") if "gbif_key" in occ_df.columns else ("taxon_id", "count"),
    n_spatial_blocks=("spatial_block_id", "nunique"),
    mean_abs_latitude=("abs_latitude", "mean"),
    lat_range=("decimal_latitude", lambda x: x.max() - x.min()),
    lon_range=("decimal_longitude", lambda x: x.max() - x.min()),
).reset_index()

taxon_compound_feature_df = taxon_compound_feature_df.merge(occ_summary, on="taxon_id", how="left")

# Numeric matrix
numeric_cols = taxon_compound_feature_df.select_dtypes(include=[np.number]).columns.tolist()
# Ensure numeric conversion for object columns that should be numeric
for c in taxon_compound_feature_df.columns:
    if c not in group_cols:
        converted = pd.to_numeric(taxon_compound_feature_df[c], errors="coerce")
        if converted.notna().sum() > 0:
            taxon_compound_feature_df[c] = converted

numeric_cols = [
    c for c in taxon_compound_feature_df.columns
    if pd.api.types.is_numeric_dtype(taxon_compound_feature_df[c]) and taxon_compound_feature_df[c].notna().sum() > 1
]

write_table(taxon_compound_feature_df, DIRS["derived"] / "taxon_compound_feature_matrix_for_relationships")
print("Taxon-compound feature matrix:", taxon_compound_feature_df.shape)
print("Numeric features:", len(numeric_cols))
taxon_compound_feature_df.head()

In [ ]:
# ============================================================
# 17B. Pearson and Spearman correlation matrices
# ============================================================

corr_input = taxon_compound_feature_df[numeric_cols].copy()
corr_input = corr_input.loc[:, corr_input.nunique(dropna=True) > 1]

pearson_corr = corr_input.corr(method="pearson")
spearman_corr = corr_input.corr(method="spearman")

write_table(pearson_corr.reset_index().rename(columns={"index": "feature"}), DIRS["derived"] / "feature_correlation_pearson")
write_table(spearman_corr.reset_index().rename(columns={"index": "feature"}), DIRS["derived"] / "feature_correlation_spearman")

# Clustered order for heatmap
corr_for_order = spearman_corr.fillna(0)
if SCIPY_AVAILABLE and corr_for_order.shape[0] > 2:
    dist = 1 - np.abs(corr_for_order.values)
    np.fill_diagonal(dist, 0)
    order = leaves_list(linkage(dist, method="average"))
else:
    order = np.arange(corr_for_order.shape[0])

ordered_features = corr_for_order.index[order]
corr_ordered = corr_for_order.loc[ordered_features, ordered_features]

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(corr_ordered.values, vmin=-1, vmax=1, interpolation="nearest")

ax.set_xticks(range(len(ordered_features)))
ax.set_yticks(range(len(ordered_features)))
ax.set_xticklabels(ordered_features, rotation=90, fontsize=7)
ax.set_yticklabels(ordered_features, fontsize=7)
ax.set_title("Clustered Spearman correlation among engineered features")
cbar = fig.colorbar(im, ax=ax, shrink=0.75)
cbar.set_label("Spearman correlation")

save_figure(fig, "figure6_clustered_feature_correlation_spearman")
plt.show()

In [ ]:
# ============================================================
# 17C. Feature–target correlation ranking
# ============================================================

target_col = "target_evidence_score"
if target_col in corr_input.columns:
    target_corr = (
        spearman_corr[target_col]
        .drop(labels=[target_col], errors="ignore")
        .dropna()
        .sort_values(key=lambda s: s.abs(), ascending=False)
        .reset_index()
    )
    target_corr.columns = ["feature", "spearman_correlation_with_target_evidence"]
    write_table(target_corr, DIRS["derived"] / "feature_target_spearman_correlation_ranking")
    display(target_corr.head(25))

    top_features = target_corr.head(8)["feature"].tolist()

    n = len(top_features)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, feat in zip(axes, top_features):
        x = pd.to_numeric(taxon_compound_feature_df[feat], errors="coerce")
        y = pd.to_numeric(taxon_compound_feature_df[target_col], errors="coerce")
        ax.scatter(x, y, alpha=0.65, s=40)
        ax.set_xlabel(feat)
        ax.set_ylabel("Target evidence score")
        ax.set_title(f"{feat} vs target evidence")
        ax.grid(True, alpha=0.25)

    for ax in axes[len(top_features):]:
        ax.axis("off")

    fig.suptitle("Top feature relationships with medicinal target-evidence score", y=1.02)
    save_figure(fig, "figure7_top_feature_relationships_with_target_evidence")
    plt.show()
else:
    print("target_evidence_score not found in correlation input.")

## 18. PCA analysis of engineered features

PCA helps show whether the integrated open-data feature space separates taxa/compounds by medicinal-evidence strength and by data support.

Interpretation:

- PCA is unsupervised.
- Strong separation can support the claim that the integrated feature design captures biologically meaningful structure.
- It is not a classifier and should not be interpreted as causal evidence.

In [ ]:
# ============================================================
# 18A. PCA on taxon-compound feature matrix
# ============================================================

# Select numeric features, excluding direct labels to avoid circularity in unsupervised feature PCA.
label_like = {
    "target_evidence_score",
    "target_label_for_open_data_classifier",
    "n_supporting_pubmed_records",
}
pca_features = [c for c in numeric_cols if c not in label_like]
pca_features = [c for c in pca_features if taxon_compound_feature_df[c].notna().sum() > 1 and taxon_compound_feature_df[c].nunique(dropna=True) > 1]

X_raw = taxon_compound_feature_df[pca_features].copy()
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X = imputer.fit_transform(X_raw)
X = scaler.fit_transform(X)

n_components = min(5, X.shape[1], X.shape[0])
pca = PCA(n_components=n_components, random_state=CONFIG["random_seed"])
scores = pca.fit_transform(X)

pca_scores_df = taxon_compound_feature_df[group_cols].copy()
for i in range(n_components):
    pca_scores_df[f"PC{i+1}"] = scores[:, i]
pca_scores_df["target_evidence_score"] = taxon_compound_feature_df.get("target_evidence_score", np.nan)
pca_scores_df["target_evidence_class"] = taxon_compound_feature_df.get("target_evidence_class", "")

loadings_df = pd.DataFrame(
    pca.components_.T,
    index=pca_features,
    columns=[f"PC{i+1}" for i in range(n_components)]
).reset_index().rename(columns={"index": "feature"})

explained_df = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(n_components)],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
})

write_table(pca_scores_df, DIRS["derived"] / "pca_scores_taxon_compound_features")
write_table(loadings_df, DIRS["derived"] / "pca_loadings_taxon_compound_features")
write_table(explained_df, DIRS["derived"] / "pca_explained_variance_taxon_compound_features")

display(explained_df)
display(loadings_df.reindex(loadings_df["PC1"].abs().sort_values(ascending=False).index).head(15))

In [ ]:
# ============================================================
# 18B. PCA biplot
# ============================================================

if n_components >= 2:
    fig, ax = plt.subplots(figsize=(10, 8))

    # Color by target evidence score
    color_vals = pd.to_numeric(pca_scores_df.get("target_evidence_score", pd.Series(np.zeros(len(pca_scores_df)))), errors="coerce").fillna(0)
    sc = ax.scatter(pca_scores_df["PC1"], pca_scores_df["PC2"], c=color_vals, s=70, alpha=0.75)

    # Label curated/strong points or small datasets
    for _, r in pca_scores_df.iterrows():
        if r.get("target_evidence_score", 0) >= 2:
            label = f"{r.get('scientific_name', '')} / {r.get('compound_name', '')}"
            ax.text(r["PC1"] + 0.02, r["PC2"] + 0.02, label, fontsize=7)

    # Loadings arrows for top features
    loading = loadings_df.copy()
    loading["loading_strength"] = np.sqrt(loading["PC1"]**2 + loading["PC2"]**2)
    top_loadings = loading.sort_values("loading_strength", ascending=False).head(10)

    x_span = pca_scores_df["PC1"].max() - pca_scores_df["PC1"].min()
    y_span = pca_scores_df["PC2"].max() - pca_scores_df["PC2"].min()
    arrow_scale = 0.35 * min(x_span if x_span > 0 else 1, y_span if y_span > 0 else 1)

    for _, r in top_loadings.iterrows():
        ax.arrow(0, 0, r["PC1"] * arrow_scale, r["PC2"] * arrow_scale,
                 head_width=0.04 * arrow_scale, length_includes_head=True, alpha=0.75)
        ax.text(r["PC1"] * arrow_scale * 1.12, r["PC2"] * arrow_scale * 1.12, r["feature"], fontsize=8)

    ax.axhline(0, linewidth=0.8, alpha=0.5)
    ax.axvline(0, linewidth=0.8, alpha=0.5)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
    ax.set_title("PCA biplot of integrated open-data feature space")
    cbar = fig.colorbar(sc, ax=ax, shrink=0.8)
    cbar.set_label("Target evidence score")

    save_figure(fig, "figure8_pca_biplot_integrated_open_data_features")
    plt.show()
else:
    print("Not enough features/components for PCA biplot.")

## 19. PCA of occurrence/environmental space

This PCA uses occurrence-level geographic and environmental proxy features. If WorldClim/SoilGrids extraction is enabled, their variables are included automatically.

In [ ]:
# ============================================================
# 19. Occurrence/environment PCA
# ============================================================

env_feature_candidates = [
    "decimal_latitude", "decimal_longitude", "abs_latitude", "latitude_squared",
    "longitude_squared", "sin_latitude", "cos_latitude", "sin_longitude", "cos_longitude",
    "coordinate_uncertainty_log1p"
]
env_feature_candidates += [c for c in ml_table.columns if c.startswith("worldclim_bio")]
env_feature_candidates += [c for c in ml_table.columns if c.startswith("soilgrids_") and c.endswith("_mean")]
env_feature_candidates = [c for c in env_feature_candidates if c in occ_df.columns or c in ml_table.columns]

# Build occurrence-level table, using occ_df for quick features.
env_df = occ_df[["gbif_key", "taxon_id", "decimal_latitude", "decimal_longitude", "spatial_block_id"] + [c for c in env_feature_candidates if c in occ_df.columns and c not in ["decimal_latitude", "decimal_longitude"]]].copy()
env_numeric_cols = [c for c in env_feature_candidates if c in env_df.columns]
env_numeric_cols = [c for c in env_numeric_cols if pd.to_numeric(env_df[c], errors="coerce").notna().sum() > 1]

env_for_pca = env_df[env_numeric_cols].apply(pd.to_numeric, errors="coerce")
env_for_pca = env_for_pca.loc[:, env_for_pca.nunique(dropna=True) > 1]

if env_for_pca.shape[1] >= 2 and len(env_for_pca) >= 3:
    env_X = SimpleImputer(strategy="median").fit_transform(env_for_pca)
    env_X = StandardScaler().fit_transform(env_X)

    env_pca = PCA(n_components=min(3, env_X.shape[1], env_X.shape[0]), random_state=CONFIG["random_seed"])
    env_scores = env_pca.fit_transform(env_X)

    env_pca_scores_df = env_df[["gbif_key", "taxon_id", "decimal_latitude", "decimal_longitude", "spatial_block_id"]].copy()
    for i in range(env_scores.shape[1]):
        env_pca_scores_df[f"ENV_PC{i+1}"] = env_scores[:, i]

    env_loadings_df = pd.DataFrame(
        env_pca.components_.T,
        index=env_for_pca.columns,
        columns=[f"ENV_PC{i+1}" for i in range(env_scores.shape[1])]
    ).reset_index().rename(columns={"index": "feature"})

    write_table(env_pca_scores_df, DIRS["derived"] / "environment_pca_scores_by_occurrence")
    write_table(env_loadings_df, DIRS["derived"] / "environment_pca_loadings")

    # Plot
    fig, ax = plt.subplots(figsize=(9, 7))
    taxa = env_pca_scores_df["taxon_id"].unique().tolist()
    for taxon_id in taxa:
        sub = env_pca_scores_df[env_pca_scores_df["taxon_id"] == taxon_id]
        ax.scatter(sub["ENV_PC1"], sub["ENV_PC2"], s=10, alpha=0.35, label=taxon_id)

    ax.set_xlabel(f"Environment PC1 ({env_pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"Environment PC2 ({env_pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_title("Occurrence-level environmental/geographic PCA")
    if len(taxa) <= 12:
        ax.legend(frameon=False, fontsize=7, loc="center left", bbox_to_anchor=(1.02, 0.5))
    save_figure(fig, "figure9_environmental_space_pca_by_occurrence")
    plt.show()

    display(env_loadings_df)
else:
    env_pca_scores_df = pd.DataFrame()
    env_loadings_df = pd.DataFrame()
    print("Not enough environmental features for PCA.")

## 20. Publication Figure 10 — Integrated taxon–compound–pathway graph

This graph is a visual summary of the manuscript’s conceptual novelty: taxa, compounds, pathways and gene families are represented as connected evidence layers.

The graph is not used for ML yet. It defines the graph structure for a later GNN/Graph Transformer notebook.

In [ ]:
# ============================================================
# 20. Integrated graph visualization
# ============================================================

if NETWORKX_AVAILABLE:
    G = nx.Graph()

    # Select manageable high-confidence subsets for visualization.
    taxa_vis = open_genetic_df[
        ~open_genetic_df.get("open_data_ml_readiness", "").eq("hierarchy_context")
    ].copy() if "open_data_ml_readiness" in open_genetic_df else open_genetic_df.copy()

    for _, r in taxa_vis.iterrows():
        G.add_node(r["taxon_id"], label=r.get("scientific_name", r["taxon_id"]), node_type="taxon")

    # Add compounds with evidence score >= 2 or curated positives.
    target_vis = target_df[target_df["target_evidence_score"] >= 2].copy()
    for _, r in target_vis.iterrows():
        G.add_node(r["compound_id"], label=r["compound_name"], node_type="compound")
        G.add_edge(r["taxon_id"], r["compound_id"], edge_type="taxon_compound", weight=1 + r["target_evidence_score"])

    # Add pathways with nonzero potential for visible taxa
    pwy_vis = pathway_df[pathway_df["taxon_id"].isin(taxa_vis["taxon_id"])].copy()
    if "pathway_potential_score" in pwy_vis:
        pwy_vis["pathway_potential_score"] = pd.to_numeric(pwy_vis["pathway_potential_score"], errors="coerce").fillna(0)
        pwy_vis = pwy_vis[pwy_vis["pathway_potential_score"] > 0]

    for _, r in pwy_vis.iterrows():
        G.add_node(r["pathway_id"], label=r["pathway_id"].replace("PWY_", ""), node_type="pathway")
        G.add_edge(r["taxon_id"], r["pathway_id"], edge_type="taxon_pathway", weight=1 + min(float(r.get("pathway_potential_score", 0)), 10) / 5)

    # Pathway-compound links from CONFIG where possible
    pathway_to_compounds = {
        "PWY_SALIDROSIDE": ["CMP_SALIDROSIDE", "CMP_TYROSOL"],
        "PWY_HYPERICIN": ["CMP_HYPERICIN"],
        "PWY_TERPENOID": ["CMP_ARTEMISININ", "CMP_PACLITAXEL", "CMP_GLYCYRRHIZIN", "CMP_TERPENOIDS"],
        "PWY_PHENYLPROPANOID_FLAVONOID": ["CMP_TOTAL_FLAVONOIDS", "CMP_TOTAL_PHENOLICS"],
        "PWY_ALKALOID": ["CMP_ALKALOIDS"],
    }
    for pwy, comps in pathway_to_compounds.items():
        if pwy in G:
            for comp in comps:
                if comp in G:
                    G.add_edge(pwy, comp, edge_type="pathway_compound", weight=1)

    node_types = nx.get_node_attributes(G, "node_type")
    type_to_marker_size = {"taxon": 900, "compound": 700, "pathway": 600}
    type_to_shape_order = ["taxon", "compound", "pathway"]

    pos = nx.spring_layout(G, seed=CONFIG["random_seed"], k=0.75)

    fig, ax = plt.subplots(figsize=(13, 10))
    for node_type in type_to_shape_order:
        nodelist = [n for n, t in node_types.items() if t == node_type]
        nx.draw_networkx_nodes(
            G, pos, nodelist=nodelist,
            node_size=type_to_marker_size.get(node_type, 500),
            alpha=0.85,
            label=node_type,
            ax=ax
        )

    widths = [max(0.5, G[u][v].get("weight", 1) * 0.6) for u, v in G.edges()]
    nx.draw_networkx_edges(G, pos, width=widths, alpha=0.35, ax=ax)

    labels = nx.get_node_attributes(G, "label")
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, ax=ax)

    ax.set_title("Integrated open-evidence graph: taxa, medicinal compounds and biosynthetic pathways")
    ax.axis("off")
    ax.legend(frameon=False, loc="upper left")

    save_figure(fig, "figure10_integrated_taxon_compound_pathway_graph")
    plt.show()

    # Save graph edge/node tables used in this visualization
    graph_vis_nodes = pd.DataFrame([
        {"node_id": n, "label": labels.get(n, n), "node_type": node_types.get(n, "")}
        for n in G.nodes()
    ])
    graph_vis_edges = pd.DataFrame([
        {"source": u, "target": v, "edge_type": G[u][v].get("edge_type", ""), "weight": G[u][v].get("weight", 1)}
        for u, v in G.edges()
    ])
    write_table(graph_vis_nodes, DIRS["derived"] / "notebook2_visual_graph_nodes")
    write_table(graph_vis_edges, DIRS["derived"] / "notebook2_visual_graph_edges")
else:
    print("networkx not available. Skipping graph visualization.")

## 21. Relationship matrix for manuscript reporting

This table summarizes feature-block relationships at a high level. It can be included in supplementary material or used to select features for Notebook 3.

In [ ]:
# ============================================================
# 21. Feature-block relationship summary
# ============================================================

feature_blocks = {
    "target": ["target_evidence_score", "target_label_for_open_data_classifier", "n_supporting_pubmed_records"],
    "genetic_availability": ["genetic_data_availability_score", "n_public_barcode_records", "n_chloroplast_plastid_records", "n_public_genome_assemblies", "n_ena_rnaseq_runs"],
    "pathway": ["pathway_support_score", "n_pathways_with_specific_candidates", "sum_pathway_potential_score"],
    "occurrence_environment": ["abs_latitude", "latitude_squared", "coordinate_uncertainty_log1p", "n_occurrences", "n_spatial_blocks", "mean_abs_latitude", "lat_range", "lon_range"],
}

block_rows = []
for block_a, feats_a in feature_blocks.items():
    feats_a = [f for f in feats_a if f in taxon_compound_feature_df.columns]
    for block_b, feats_b in feature_blocks.items():
        feats_b = [f for f in feats_b if f in taxon_compound_feature_df.columns]
        if not feats_a or not feats_b:
            continue
        vals = []
        for fa in feats_a:
            for fb in feats_b:
                if fa == fb:
                    continue
                xa = pd.to_numeric(taxon_compound_feature_df[fa], errors="coerce")
                xb = pd.to_numeric(taxon_compound_feature_df[fb], errors="coerce")
                if xa.notna().sum() > 2 and xb.notna().sum() > 2 and xa.nunique(dropna=True) > 1 and xb.nunique(dropna=True) > 1:
                    vals.append(xa.corr(xb, method="spearman"))
        vals = [v for v in vals if pd.notna(v)]
        block_rows.append({
            "block_a": block_a,
            "block_b": block_b,
            "mean_abs_spearman": float(np.mean(np.abs(vals))) if vals else np.nan,
            "max_abs_spearman": float(np.max(np.abs(vals))) if vals else np.nan,
            "n_pairwise_correlations": len(vals),
        })

block_relationship_df = pd.DataFrame(block_rows)
write_table(block_relationship_df, DIRS["derived"] / "feature_block_relationship_summary")
block_relationship_df

## 22. Recommended validation split tables

These are not train/test splits yet. They are grouping variables that Notebook 3 must use to avoid data leakage.

In [ ]:
# ============================================================
# 22. Validation group tables
# ============================================================

validation_groups_df = ml_table[[
    "ml_row_id", "taxon_id", "compound_id", "spatial_group_id",
    "taxonomic_group_id", "compound_group_id", "source_group_id",
    "target_evidence_score", "pseudo_negative_flag"
]].copy()

validation_groups_df["recommended_primary_cv"] = "spatial_block_and_taxon_grouped"
validation_groups_df["leakage_warning"] = "Do not use simple random row split as primary validation."

write_table(validation_groups_df, DIRS["derived"] / "validation_group_table_for_notebook3")

# Summary counts
validation_summary = pd.DataFrame([
    {"group_type": "spatial_group_id", "n_groups": validation_groups_df["spatial_group_id"].nunique()},
    {"group_type": "taxonomic_group_id", "n_groups": validation_groups_df["taxonomic_group_id"].nunique()},
    {"group_type": "compound_group_id", "n_groups": validation_groups_df["compound_group_id"].nunique()},
    {"group_type": "source_group_id", "n_groups": validation_groups_df["source_group_id"].nunique()},
    {"group_type": "rows", "n_groups": len(validation_groups_df)},
])
write_table(validation_summary, DIRS["derived"] / "validation_group_summary")
validation_summary

## 23. Final QC report and manifest

This section writes a reproducibility manifest and a concise QC report for publication/supplementary material.

In [ ]:
# ============================================================
# 23. Final QC report and manifest
# ============================================================

qc_lines = []
qc_lines.append("# Notebook 2 QC report")
qc_lines.append(f"Run ID: {RUN_ID}")
qc_lines.append(f"Created UTC: {CONFIG['created_utc']}")
qc_lines.append("")
qc_lines.append("## Main outputs")
qc_lines.append(f"- Integrated ML table rows: {len(ml_table)}")
qc_lines.append(f"- Unique taxa in ML table: {ml_table['taxon_id'].nunique() if 'taxon_id' in ml_table else 0}")
qc_lines.append(f"- Unique compounds in ML table: {ml_table['compound_id'].nunique() if 'compound_id' in ml_table else 0}")
qc_lines.append(f"- Occurrences used: {occ_df['gbif_key'].nunique() if 'gbif_key' in occ_df else len(occ_df)}")
qc_lines.append(f"- Spatial blocks: {occ_df['spatial_block_id'].nunique() if 'spatial_block_id' in occ_df else 0}")
qc_lines.append("")
qc_lines.append("## Target evidence counts")
if "target_evidence_class" in ml_table:
    for cls, n in ml_table["target_evidence_class"].value_counts(dropna=False).items():
        qc_lines.append(f"- {cls}: {n}")
qc_lines.append("")
qc_lines.append("## Key limitations")
qc_lines.append("- Target evidence is literature/open-data evidence, not confirmed concentration unless manually curated.")
qc_lines.append("- Missing literature is not confirmed metabolite absence.")
qc_lines.append("- Open genetic evidence is usually taxon-level, not exact population-level genotype.")
qc_lines.append("- Environment features in quick mode are spatial proxies; enable WorldClim/SoilGrids for publication-grade environmental predictors.")
qc_lines.append("- Family/hierarchy rows should not be ordinary ML samples.")

qc_path = DIRS["derived"] / "notebook2_qc_report.md"
qc_path.write_text("\n".join(qc_lines), encoding="utf-8")

manifest_rows = []
for path in PROJECT.rglob("*"):
    if path.is_file():
        try:
            manifest_rows.append({
                "path": str(path.relative_to(PROJECT)),
                "absolute_path": str(path),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
                "modified_utc": dt.datetime.fromtimestamp(path.stat().st_mtime, tz=dt.timezone.utc).isoformat(timespec="seconds"),
                "run_id": RUN_ID,
            })
        except Exception as e:
            print("Skipping manifest entry", path, e)

run_manifest_df = pd.DataFrame(manifest_rows).sort_values("path")
write_table(run_manifest_df, PROJECT / "run_file_manifest")

print(qc_path)
display(run_manifest_df.head(20))

## 24. Outputs for Notebook 3 — ML training and explainability

Use these files in Notebook 3:

Primary table:

- `derived/taxon_occurrence_compound_ml_table.tsv`

Supporting tables:

- `derived/taxon_compound_feature_matrix_for_relationships.tsv`
- `derived/validation_group_table_for_notebook3.tsv`
- `derived/validation_group_summary.tsv`
- `derived/taxon_compound_target_evidence_refined.tsv`
- `derived/pathway_features_by_taxon_wide.tsv`
- `derived/feature_target_spearman_correlation_ranking.tsv`
- `derived/pca_scores_taxon_compound_features.tsv`
- `derived/pca_loadings_taxon_compound_features.tsv`

Recommended Notebook 3 models:

1. baseline logistic/ordinal model for target-evidence class;
2. random forest / gradient boosting with grouped CV;
3. positive-unlabeled or pseudo-negative sensitivity analysis;
4. SHAP or permutation importance;
5. graph model only after tabular baseline is stable.

Recommended validation:

- spatial-block holdout;
- taxon-grouped holdout;
- compound-class holdout;
- source-aware holdout;
- no random row split as the primary result.